In [1]:
# 1. Environment check
import sys
import platform
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("Python version:", sys.version)
print("Architecture:", platform.machine())
print("Current working directory:", Path.cwd())
print("Home directory:", Path.home())

for package, label in [("numpy", "NumPy version"),
                       ("tflite-runtime", "tflite-runtime version")]:
    try:
        print(f"{label}: {version(package)}")
    except PackageNotFoundError:
        print(f"{label}: package non trouve dans ce kernel")


Python version: 3.10.4 (main, Apr  2 2022, 09:04:19) [GCC 11.2.0]
Architecture: armv7l
Current working directory: /home/xilinx/jupyter_notebooks
Home directory: /root
NumPy version: 1.21.5
tflite-runtime version: 2.13.0


## 1. Environment check

La cellule precedente doit etre executee dans le **kernel Jupyter distant du PYNQ-Z2**.
Les chemins affiches appartiennent au PYNQ, pas a la VM qui stocke ce notebook.
Le chemin local du fichier `.ipynb` ne rend pas les fichiers voisins accessibles au kernel.

Environnement valide manuellement : Python 3.10.4, ARMv7 (`armv7l`),
NumPy 1.21.5 et tflite-runtime 2.13.0. Comparer avec les valeurs affichees.
Aucune installation ni aucun telechargement ne sont effectues par ce notebook.


## 2. Runtime check

Verifier que NumPy et l'interpreteur TensorFlow Lite sont importables dans ce kernel.
Un import reussi ne garantit pas encore que tous les operateurs du modele sont compatibles :
le chargement et l'allocation des tenseurs le verifieront partiellement.


In [2]:
import numpy as np
from tflite_runtime.interpreter import Interpreter

print("NumPy import: OK —", np.__version__)
print("Interpreter import: OK")
print("Runtime module:", Interpreter.__module__)


NumPy import: OK — 1.21.5
Interpreter import: OK
Runtime module: tflite_runtime.interpreter


## 3. Model configuration

Modele retenu : **SSD MobileNet V1 COCO quantifie 8 bits, export TFLite CPU**,
non compile pour Edge TPU.

Renseigner `MODEL_PATH` avec le chemin du fichier `.tflite` **accessible depuis le PYNQ**.
Un chemin absolu est preferable. `~` designe le home du kernel distant ; un chemin relatif
est interprete depuis son repertoire de travail affiche plus haut.

La valeur reste volontairement vide : aucun emplacement definitif n'est suppose.
Le modele doit etre transfere ou prepare separement sur le PYNQ.


In [3]:
MODEL_PATH = ""  # A renseigner : chemin du modele .tflite sur le PYNQ.
print("Configured model path:", repr(MODEL_PATH))


Configured model path: ''


## 4. Model file validation

Arreter explicitement le notebook si le chemin est vide, absent, designe un repertoire
ou si le fichier est vide. Ces controles portent sur le systeme de fichiers du **kernel distant**.
Ils ne prouvent pas encore que le contenu est un modele TFLite valide.


In [4]:
if not str(MODEL_PATH).strip():
    raise ValueError(
        "MODEL_PATH est vide. Renseignez le chemin du modele .tflite sur le PYNQ, "
        "puis reexecutez les cellules de configuration et de validation."
    )

MODEL_PATH = Path(MODEL_PATH).expanduser()
print("Model path:", MODEL_PATH.resolve())

if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(
        f"Modele introuvable sur le PYNQ : {MODEL_PATH.resolve()}. "
        f"Repertoire du kernel : {Path.cwd()}. "
        "Un fichier present uniquement dans le depot local de la VM "
        "n'est pas automatiquement accessible au kernel distant."
    )

if not MODEL_PATH.is_file():
    raise ValueError(f"Le chemin doit designer un fichier : {MODEL_PATH}")

model_size = MODEL_PATH.stat().st_size
if model_size == 0:
    raise ValueError(f"Le fichier modele est vide : {MODEL_PATH}")

print("Model file validation: OK")
print(f"Model size: {model_size:,} bytes ({model_size / 1024**2:.2f} MiB)")


ValueError: MODEL_PATH est vide. Renseignez le chemin du modele .tflite sur le PYNQ, puis reexecutez les cellules de configuration et de validation.

## 5. Model loading

Creer l'interpreteur puis allouer les tenseurs. Cette etape ne lance **aucune inference**.
Une erreur ici peut indiquer un fichier invalide, des operateurs incompatibles avec
le runtime 2.13.0 ou un manque de memoire. Conserver le message et la traceback pour le diagnostic.


In [ ]:
interpreter = None
try:
    interpreter = Interpreter(model_path=str(MODEL_PATH))
    interpreter.allocate_tensors()
except Exception:
    interpreter = None
    print("Model loading failed. Verifier le fichier TFLite CPU, les operateurs et la memoire.")
    raise

print("Model loading: OK")
print("Tensor allocation: OK")


## 6. Input/output tensor inspection

Afficher les caracteristiques reelles du modele avant de definir son preprocessing
ou d'interpreter ses sorties. Ne pas supposer l'ordre des tenseurs de sortie.

- `shape` : dimensions allouees ; `shape_signature` : dimensions declarees, parfois dynamiques.
- `dtype` : type des valeurs attendues ou produites.
- `quantization` : couple `(scale, zero_point)` pour la quantification par tenseur.
- `quantization_parameters` : echelles, points zero et axe, y compris la quantification par axe.

Des echelles absentes peuvent indiquer un tenseur non quantifie. Un modele aux poids
quantifies peut tout de meme exposer certaines sorties en flottants.


In [ ]:
if interpreter is None:
    raise RuntimeError("Charger et allouer le modele avant d'inspecter ses tenseurs.")

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

for group, details in [("Input", input_details), ("Output", output_details)]:
    print(f"\n{group} tensors: {len(details)}")
    for position, tensor in enumerate(details):
        print(f"\n{group} tensor #{position} (index={tensor['index']})")
        print("  name:", tensor["name"])
        print("  shape:", tensor["shape"].tolist())
        print("  shape signature:", tensor.get("shape_signature"))
        print("  dtype:", np.dtype(tensor["dtype"]).name)
        print("  quantization:", tensor["quantization"])
        print("  quantization parameters:", tensor["quantization_parameters"])


La preparation s'arrete ici. La suite portera sur une image fixe : chargement de l'image,
preprocessing adapte aux tenseurs observes, inference chronometree, interpretation des sorties,
filtrage de la classe `person`, visualisation Matplotlib des bounding boxes et resume des resultats.


In [ ]:
from pathlib import Path

MODEL_PATH = (
    Path.home()
    / "jupyter_notebooks"
    / "models"
    / "ssd_mobilenet_v1"
    / "detect.tflite"
)

LABEL_PATH = (
    Path.home()
    / "jupyter_notebooks"
    / "models"
    / "ssd_mobilenet_v1"
    / "labelmap.txt"
)

print("Model:", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

print("Labels:", LABEL_PATH)
print("Labels exist:", LABEL_PATH.exists())

Model: /root/jupyter_notebooks/models/ssd_mobilenet_v1/detect.tflite
Model exists: False
Labels: /root/jupyter_notebooks/models/ssd_mobilenet_v1/labelmap.txt
Labels exist: False


In [ ]:
from pathlib import Path

MODEL_DIR = Path(
    "/home/xilinx/jupyter_notebooks/models/ssd_mobilenet_v1"
)

MODEL_PATH = MODEL_DIR / "detect.tflite"
LABEL_PATH = MODEL_DIR / "labelmap.txt"

print("Model:", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

print("Labels:", LABEL_PATH)
print("Labels exist:", LABEL_PATH.exists())

Model: /home/xilinx/jupyter_notebooks/models/ssd_mobilenet_v1/detect.tflite
Model exists: True
Labels: /home/xilinx/jupyter_notebooks/models/ssd_mobilenet_v1/labelmap.txt
Labels exist: True


In [ ]:
from tflite_runtime.interpreter import Interpreter

interpreter = Interpreter(model_path=str(MODEL_PATH))
interpreter.allocate_tensors()

print("Interpreter loaded: OK")

Interpreter loaded: OK


In [ ]:
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("=== INPUT ===")

for detail in input_details:
    print("name:", detail["name"])
    print("shape:", detail["shape"])
    print("dtype:", detail["dtype"])
    print("quantization:", detail["quantization"])
    print("index:", detail["index"])
    print()

print("=== OUTPUTS ===")

for i, detail in enumerate(output_details):
    print(f"Output {i}")
    print("name:", detail["name"])
    print("shape:", detail["shape"])
    print("dtype:", detail["dtype"])
    print("quantization:", detail["quantization"])
    print("index:", detail["index"])
    print()

=== INPUT ===
name: normalized_input_image_tensor
shape: [  1 300 300   3]
dtype: <class 'numpy.uint8'>
quantization: (0.0078125, 128)
index: 175

=== OUTPUTS ===
Output 0
name: TFLite_Detection_PostProcess
shape: [ 1 10  4]
dtype: <class 'numpy.float32'>
quantization: (0.0, 0)
index: 167

Output 1
name: TFLite_Detection_PostProcess:1
shape: [ 1 10]
dtype: <class 'numpy.float32'>
quantization: (0.0, 0)
index: 168

Output 2
name: TFLite_Detection_PostProcess:2
shape: [ 1 10]
dtype: <class 'numpy.float32'>
quantization: (0.0, 0)
index: 169

Output 3
name: TFLite_Detection_PostProcess:3
shape: [1]
dtype: <class 'numpy.float32'>
quantization: (0.0, 0)
index: 170



In [ ]:
for i, detail in enumerate(output_details):
    print(
        f"{i}: "
        f"name={detail['name']} | "
        f"shape={detail['shape'].tolist()} | "
        f"dtype={detail['dtype']} | "
        f"index={detail['index']}"
    )

0: name=TFLite_Detection_PostProcess | shape=[1, 10, 4] | dtype=<class 'numpy.float32'> | index=167
1: name=TFLite_Detection_PostProcess:1 | shape=[1, 10] | dtype=<class 'numpy.float32'> | index=168
2: name=TFLite_Detection_PostProcess:2 | shape=[1, 10] | dtype=<class 'numpy.float32'> | index=169
3: name=TFLite_Detection_PostProcess:3 | shape=[1] | dtype=<class 'numpy.float32'> | index=170


## 7. Input image loading

Ces cellules utilisent le kernel distant du PYNQ. Executer auparavant les cellules manuelles de chargement de detect.tflite et d'inspection des tenseurs. Les chemins ci-dessous sont ceux du PYNQ, pas ceux du depot local. Aucun acces camera n'est effectue.


In [ ]:
from pathlib import Path
import time
import cv2
import numpy as np

IMAGE_PATH = Path(
    "/home/xilinx/jupyter_notebooks/images/person_test.jpg"
)
LABEL_PATH = Path(
    "/home/xilinx/jupyter_notebooks/models/ssd_mobilenet_v1/labelmap.txt"
)
print("Image path:", IMAGE_PATH)
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Image introuvable sur le PYNQ : {IMAGE_PATH}")
if not IMAGE_PATH.is_file():
    raise ValueError(f"Le chemin ne designe pas un fichier : {IMAGE_PATH}")
image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise ValueError(f"OpenCV ne peut pas decoder l'image : {IMAGE_PATH}")
original_bgr = image_bgr.copy()
original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
original_height, original_width = original_bgr.shape[:2]
print(f"Original image: {original_width} x {original_height}")

Image path: /home/xilinx/jupyter_notebooks/images/person_test.jpg
Original image: 640 x 480


## 8. Preprocessing

Conversion BGR vers RGB, resize direct 300 × 300 et ajout du batch. Les pixels restent uint8 : aucune normalisation float. L'image originale reste intacte.


In [ ]:
if len(input_details) != 1:
    raise ValueError("Un seul tenseur d'entree est attendu.")
if tuple(input_details[0]["shape"]) != (1, 300, 300, 3):
    raise ValueError(f"Input shape inattendue : {input_details[0]['shape']}")
if np.dtype(input_details[0]["dtype"]) != np.dtype(np.uint8):
    raise TypeError("Le modele doit attendre des pixels uint8.")
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
resized_rgb = cv2.resize(image_rgb, (300, 300), interpolation=cv2.INTER_LINEAR)
input_tensor = np.expand_dims(resized_rgb, axis=0)
assert input_tensor.shape == (1, 300, 300, 3)
assert input_tensor.dtype == np.uint8
print("Input shape:", input_tensor.shape)
print("Input dtype:", input_tensor.dtype)


Input shape: (1, 300, 300, 3)
Input dtype: uint8


## 9. Inference

Les quatre sorties validees sont boxes, classes, scores et nombre de detections. Les indices sont lus dans output_details. Le temps mesure couvre uniquement invoke(), sans preprocessing ni lecture des sorties ; une seule inference est executee.


In [ ]:
expected_shapes = [(1, 10, 4), (1, 10), (1, 10), (1,)]
if len(output_details) != 4:
    raise ValueError("Quatre sorties SSD sont attendues.")
for detail, expected in zip(output_details, expected_shapes):
    if tuple(detail["shape"]) != expected:
        raise ValueError(f"Output shape inattendue pour {detail['name']}: {detail['shape']}")

interpreter.set_tensor(input_details[0]["index"], input_tensor)
start_time = time.perf_counter()
interpreter.invoke()
inference_time_ms = (time.perf_counter() - start_time) * 1000

boxes = interpreter.get_tensor(output_details[0]["index"])[0]
classes = interpreter.get_tensor(output_details[1]["index"])[0]
scores = interpreter.get_tensor(output_details[2]["index"])[0]
count_value = float(interpreter.get_tensor(output_details[3]["index"])[0])
if not np.isfinite(count_value) or not count_value.is_integer():
    raise ValueError(f"Nombre de detections invalide : {count_value}")
num_detections = int(count_value)
if not 0 <= num_detections <= min(len(boxes), len(classes), len(scores)):
    raise ValueError(f"Nombre de detections hors limites : {num_detections}")
print(f"Inference time: {inference_time_ms:.2f} ms")
print("Total detections returned:", num_detections)


: 

: 

: 

## 10. Person filtering

Retirer uniquement la premiere entree si elle vaut ???, sans decaler les autres labels. La classe 0 doit correspondre a person. Garder les personnes avec score ≥ 0.5. Les boxes normalisees [ymin, xmin, ymax, xmax] sont converties dans les dimensions originales ; x2/y2 designent les bords et peuvent atteindre largeur/hauteur.


In [ ]:
CONFIDENCE_THRESHOLD = 0.5
if not LABEL_PATH.is_file():
    raise FileNotFoundError(f"Label map introuvable sur le PYNQ : {LABEL_PATH}")
labels = [line.strip() for line in LABEL_PATH.read_text(encoding="utf-8-sig").splitlines()]
if labels and labels[0] == "???":
    labels = labels[1:]
if not labels or labels[0] != "person":
    raise ValueError("Label map incompatible : la classe 0 doit etre person.")
if not 0 <= CONFIDENCE_THRESHOLD <= 1:
    raise ValueError("Le seuil doit etre compris entre 0 et 1.")

persons = []
for i in range(num_detections):
    class_value = float(classes[i])
    score = float(scores[i])
    if not np.isfinite(class_value) or not class_value.is_integer():
        continue
    class_id = int(class_value)
    if not 0 <= class_id < len(labels):
        continue
    label = labels[class_id]
    if label != "person" or not np.isfinite(score) or score < CONFIDENCE_THRESHOLD:
        continue
    if not np.all(np.isfinite(boxes[i])):
        continue
    ymin, xmin, ymax, xmax = np.clip(boxes[i], 0.0, 1.0)
    x1 = int(np.floor(xmin * original_width))
    y1 = int(np.floor(ymin * original_height))
    x2 = int(np.ceil(xmax * original_width))
    y2 = int(np.ceil(ymax * original_height))
    if x2 <= x1 or y2 <= y1:
        continue
    persons.append({
        "label": label, "confidence": score,
        "bbox": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
    })
print("Persons above threshold:", len(persons))


## 11. Bounding box visualization

Afficher l'image originale RGB avec un rectangle, le label et la confiance pour chaque personne. Si aucune detection ne depasse le seuil, l'image est affichee sans rectangle.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(original_rgb)
for person in persons:
    box = person["bbox"]
    ax.add_patch(Rectangle(
        (box["x1"], box["y1"]),
        box["x2"] - box["x1"], box["y2"] - box["y1"],
        linewidth=2, edgecolor="lime", facecolor="none",
    ))
    ax.text(
        box["x1"], box["y1"],
        f"{person['label']} {person['confidence']:.1%}",
        color="black", va="top",
        bbox={"facecolor": "lime", "alpha": 0.8, "pad": 2},
    )
ax.set_title(f"Persons above threshold: {len(persons)}")
ax.axis("off")
plt.show()


## 12. Results summary

Resume de cette inference sur image fixe. Les bounding boxes sont exprimees en pixels de l'image originale.


In [ ]:
print(f"Inference time: {inference_time_ms:.2f} ms")
print(f"Total detections returned: {num_detections}")
print(f"Persons above threshold: {len(persons)}")
for number, person in enumerate(persons, start=1):
    box = person["bbox"]
    print(f"\nPerson {number}")
    print(f"label: {person['label']}")
    print(f"confidence: {person['confidence']:.1%}")
    print(f"bbox: x1={box['x1']}, y1={box['y1']}, x2={box['x2']}, y2={box['y2']}")
if not persons:
    print("\nAucune personne detectee au-dessus du seuil configure.")
